In [1]:
from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from src.rset_opt import *
from FasterRisk.src.fasterrisk import fasterrisk
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *

import pandas as pd
from time import time

### Swapping Method

In [ ]:
dataset_settings = {
    'bank': {
        'gap_tolerance': 0.006,
        'num_estimators': 50,
    },
    'compas': {
        'gap_tolerance': 0.0025,
        'num_estimators': 50,
    },
    'diabetes': {
        'gap_tolerance': 0.005,
        'num_estimators': 200,
    },
    'netherlands': {
        'gap_tolerance': 0.0015,
        'num_estimators': 50,
    },
    'spambase': {
        'gap_tolerance': 0.01,
        'num_estimators': 50,
    },
    'mimic2': {
        'gap_tolerance':0.002,
        'num_estimators': 50,
    },
}

results = []
for dataset_name, settings in dataset_settings.items():
    ne = settings["num_estimators"]
    gt = settings['gap_tolerance']

    path = 'datasets/{}.csv'.format(dataset_name)
    dataset = pd.read_csv(path)
    print(f"Dataset: {dataset_name}")
    print(f"Binarized shape: {dataset.shape}")

    df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, ne)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]
    header = pd.Index(["intercept"] + list(X.columns)).astype("object")
    X_one_hot, y = utils.get_X_y(X, y)

    start = time()
    rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=15, lb=-100, ub=100, gap_tolerance=gt, select_top_m=-1, maxAttempts=25)
    rs.optimize_with_swaps_beam_search(swaps=5, beam_size=100, verbose=True)
    end = time()

    print(f"\t{rs.sparseDiversePool_betas.shape[0]} solutions, {end - start:.2f} seconds")
    results.append({
        "dataset": dataset_name,
        "dataset_shape": dataset.shape,
        "num_estimators": ne,
        "gap_tolerance": gt,
        "threshold_guess_time": threshold_guess_time,
        "num_features": len(header),
        "runtime": end - start,
        "betas": rs.sparseDiversePool_betas,
        "beta0": rs.sparseDiversePool_beta0,
        "num_solutions": rs.sparseDiversePool_betas.shape[0],
        "loss": get_loss(X_one_hot, y, rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas),
        "predictions": get_predictions(X_one_hot, rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas),
    })

with open(f"analysis/results/swapping_method.pkl", "wb") as f:
    pickle.dump(results, f)

Dataset: bank
Binarized shape: (4521, 17)
swap 0, beam size 1
swap 1, beam size 49
swap 2, beam size 100
swap 3, beam size 100
swap 4, beam size 100
	100 solutions, 356.23 seconds
Dataset: compas
Binarized shape: (6907, 8)
swap 0, beam size 1
swap 1, beam size 58
swap 2, beam size 100
swap 3, beam size 100
swap 4, beam size 100
	100 solutions, 296.97 seconds
Dataset: diabetes
Binarized shape: (768, 9)
swap 0, beam size 1
swap 1, beam size 15
swap 2, beam size 47
swap 3, beam size 44
swap 4, beam size 55
	100 solutions, 13.45 seconds
Dataset: netherlands
Binarized shape: (20000, 10)
swap 0, beam size 1
swap 1, beam size 56
swap 2, beam size 100
swap 3, beam size 100
swap 4, beam size 100
	100 solutions, 1206.28 seconds
Dataset: spambase
Binarized shape: (4601, 58)
swap 0, beam size 1
swap 1, beam size 14
swap 2, beam size 65
swap 3, beam size 100
swap 4, beam size 100
	37 solutions, 84.42 seconds
Dataset: mimic2
Binarized shape: (24508, 18)
swap 0, beam size 1
swap 1, beam size 16
swap 

### Ellipsoid Sampling

In [3]:
dataset_settings = {
    'bank': {
        "l0": 0.001,
        "l2": 0.5,
        "m": 0.01,
    },
    'compas': {
        "l0": 0.001,
        "l2": 0.5,
        "m": 0.01,
    },
    "diabetes": {
        "l0": 0.001,
        "l2": 0.5,
        "m": 0.01,
    },
    # 'netherlands': {
    #     "l0": 0.001,
    #     "l2": 0.5,
    #     "m": 0.01,
    # },
    'spambase': {
        "l0": 0.001,
        "l2": 0.5,
        "m": 0.01,
    },
    'mimic2': {
        "l0": 0.0005,
        "l2": 0.001,
        "m": 1.01,
    }
}

methods = [
    {"method": "uniform"},
    {"method": "poisson", "r_min": 1, "max_attempts": 1000},
]

for method in methods:
    results = []
    for dname, settings in dataset_settings.items():
        l0 = settings["l0"]
        l2 = settings["l2"]
        m = settings["m"]

        betas_fastSparse = prepare_sparse_gam(dname, lamb0=l0, lamb2=l2, multiplier=m)
        filepath = "{}_{}_{}_{}.p".format(dname, l0, l2, m)

        w_samples = get_models_from_rset(filepath, n_samples=100, plot_shape=False, sample_from_surface=False, method=method)
        X = pickle.load(open(betas_fastSparse, 'rb'))['X']

        results.append({
            "dataset": dname,
            "l0": l0,
            "l2": l2,
            "m": m,
            "w_samples": w_samples,
            "predictions": get_predictions(X, np.zeros(len(w_samples)), w_samples),
        })

    with open(f"""analysis/results/{method["method"]}_method.pkl""", "wb") as f:
        pickle.dump(results, f)

header dimension 3719
lamb0:0.001, lamb2:0.5, acc:0.8847600088476001, auc:0.8309760076775431, supp_size:19
pair before removing {'month': [9.0], 'duration': [210.0, 211.0, 212.0, 220.0, 221.0, 248.0, 342.0, 343.0, 383.0, 414.0, 631.0, 638.0, 640.0, 643.0, 646.0, 758.0], 'pdays': [28.0]}
k, v month [9.0, 11]
k, v duration [210.0, 211.0, 212.0, 220.0, 221.0, 248.0, 342.0, 343.0, 383.0, 414.0, 631.0, 638.0, 640.0, 643.0, 646.0, 758.0, 3025]
k, v pdays [28.0, 871]
pair after removing {'month': [9.0, 11], 'duration': [210.0, 211.0, 212.0, 220.0, 221.0, 248.0, 342.0, 343.0, 383.0, 414.0, 631.0, 638.0, 640.0, 643.0, 646.0, 758.0, 3025], 'pdays': [28.0, 871]}
v_count: [0, 2, 17, 2]
[ 0  2 19 21]
month 0 0
month 1 1
duration 0 2
duration 1 3
duration 2 4
duration 3 5
duration 4 6
duration 5 7
duration 6 8
duration 7 9
duration 8 10
duration 9 11
duration 10 12
duration 11 13
duration 12 14
duration 13 15
duration 14 16
duration 15 17
duration 16 18
pdays 0 19
pdays 1 20
True
True
True
(22,) hi


### Ellipsoid Zeroing

### Comparison

In [9]:
methods = ["swapping_method", "uniform_method", "poisson_method"]
for method in methods:
    with open(f"analysis/results/{method}.pkl", "rb") as f:
        results = pickle.load(f)

    print(f"Results for {method}:")
    for result in results:
        dataset_name = result["dataset"]
        predictions = result["predictions"]
        print(f"\tDataset: {dataset_name}, predictions shape: {predictions.shape}")

Results for swapping_method:
	Dataset: bank, predictions shape: (4521, 100)
	Dataset: compas, predictions shape: (6907, 100)
	Dataset: diabetes, predictions shape: (768, 100)
	Dataset: netherlands, predictions shape: (20000, 100)
	Dataset: spambase, predictions shape: (4601, 37)
	Dataset: mimic2, predictions shape: (24508, 100)
Results for uniform_method:
	Dataset: bank, predictions shape: (4521, 100)
	Dataset: compas, predictions shape: (6907, 100)
	Dataset: diabetes, predictions shape: (768, 100)
	Dataset: spambase, predictions shape: (4601, 100)
	Dataset: mimic2, predictions shape: (24508, 100)
Results for poisson_method:
	Dataset: bank, predictions shape: (4521, 100)
	Dataset: compas, predictions shape: (6907, 100)
	Dataset: diabetes, predictions shape: (768, 100)
	Dataset: spambase, predictions shape: (4601, 100)
	Dataset: mimic2, predictions shape: (24508, 100)
